In [58]:
import cv2
import glob
import numpy as np
import shutil
from tensorflow.keras.preprocessing.image import img_to_array, ImageDataGenerator
from modules.setting import (
    CHARACTER_PATH,
    OUTPUT_PATH,
    chara_folders,
    aug_folders,
)
from sklearn.model_selection import train_test_split
from keras.utils import np_utils
from tqdm import tqdm

In [59]:
####################変更箇所####################
# 画像の拡張子(.jpg, .pngなど大文字小文字区別する)
IMAGE_EXT = ".JPG"
# 画像の縮小倍率
SCALE = 0.2
# 1画像あたりに生成する画像の枚数
AUGMENTATION_NUM = 80

data_generator = ImageDataGenerator(
    zoom_range=[1.0, 1.3],
    rotation_range=20,
    shear_range=34.3,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode="nearest",
    vertical_flip=False,
    horizontal_flip=False,
)

In [60]:
x_test = []
y_test = []
for idx, folder in enumerate(
    tqdm(chara_folders, desc="テストデータの分割・保存とトレーニング+バリデーションデータの拡張")
):
    files = glob.glob(f"{CHARACTER_PATH}/{folder}/*{IMAGE_EXT}")
    # トレーニング+バリデーション:テスト = 8:2
    train_val_data, test_data = train_test_split(
        files, test_size=0.2, random_state=2023
    )
    # test_dataに対して画像のグレースケール化、リサイズ、保存を行い、x_test, y_testを作成
    for i, file in enumerate(test_data):
        img = cv2.imread(file)
        # グレースケール化
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # リサイズ
        img = cv2.resize(img, (256, 256))
        img = cv2.resize(img, (int(img.shape[1] * SCALE), int(img.shape[0] * SCALE)))
        # 画像の保存
        cv2.imwrite(f"{CHARACTER_PATH}/test/chara{idx}_{i:03d}{IMAGE_EXT}", img)
        # numpy配列に変換
        img_array = img_to_array(img)
        x_test.append(img_array)
        y_test.append(idx)

    # train_val_dataに対して画像のグレースケール化、リサイズ、拡張を行い保存
    for i, file in enumerate(train_val_data):
        save_num = 0
        img = cv2.imread(file)
        # グレースケール化
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # リサイズ
        img = cv2.resize(img, (256, 256))
        img = cv2.resize(img, (int(img.shape[1] * SCALE), int(img.shape[0] * SCALE)))
        # numpy配列に変換
        img_array = img_to_array(img)
        for aug_idx in range(AUGMENTATION_NUM):
            aug_img = data_generator.random_transform(img_array)
            # 拡張画像をjpg形式で保存
            cv2.imwrite(
                f"{CHARACTER_PATH}/{aug_folders[idx]}/{aug_folders[idx]}_{i:03d}_{aug_idx:03d}.jpg",
                aug_img,
            )

# テストデータのnumpy配列化

x_test = np.asarray(x_test)
y_test = np.asarray(y_test)

# 信号強度の標準化
x_test = x_test.astype("float32")
x_test /= 255

# One-hot表現化
y_test = np_utils.to_categorical(y_test, 4)
# テストデータの保存
np.save(f"{OUTPUT_PATH}/x_test.npy", x_test)
np.save(f"{OUTPUT_PATH}/y_test.npy", y_test)

テストデータの分割・保存とトレーニング+バリデーションデータの拡張: 100%|█████████████████| 4/4 [00:18<00:00,  4.75s/it]


In [61]:
x_train = []
y_train = []
x_val = []
y_val = []
for idx, folder in enumerate(tqdm(aug_folders, desc="トレーニング+バリデーションデータの分割・保存")):
    files = glob.glob(f"{CHARACTER_PATH}/{folder}/*.jpg")
    # トレーニング:バリデーション = 8:2
    train_data, val_data = train_test_split(files, test_size=0.2, random_state=2023)
    for file in train_data:
        # 前処理済み画像のnumpy配列化
        img = cv2.imread(file)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img_to_array(img)
        x_train.append(img)
        y_train.append(idx)
    for file in val_data:
        # 前処理済み画像のnumpy配列化
        img = cv2.imread(file)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = img_to_array(img)
        x_val.append(img)
        y_val.append(idx)

x_train = np.asarray(x_train)
y_train = np.asarray(y_train)
x_val = np.asarray(x_val)
y_val = np.asarray(y_val)

# 信号強度の標準化
x_train = x_train.astype("float32")
x_train /= 255
x_val = x_val.astype("float32")
x_val /= 255

# One-hot表現化
y_train = np_utils.to_categorical(y_train, 4)
y_val = np_utils.to_categorical(y_val, 4)

# トレーニングデータの保存
np.save(f"{OUTPUT_PATH}/x_train.npy", x_train)
np.save(f"{OUTPUT_PATH}/y_train.npy", y_train)
# バリデーションデータの保存
np.save(f"{OUTPUT_PATH}/x_val.npy", x_val)
np.save(f"{OUTPUT_PATH}/y_val.npy", y_val)

トレーニング+バリデーションデータの分割・保存: 100%|█████████████████████████████████████| 4/4 [01:36<00:00, 24.09s/it]


In [62]:
# 結果の表示
print(
    f"訓練データ:{x_train.shape[0]}個, バリデーションデータ:{x_val.shape[0]}個, テストデータ:{x_test.shape[0]}個"
)
print(f"画像サイズ:{x_train.shape[1]}×{x_train.shape[2]}")

訓練データ:4096個, バリデーションデータ:1024個, テストデータ:16個
画像サイズ:51×51


In [56]:
def ask_delete_aug_files():
    while True:
        delete_files = input("拡張画像を削除しますか？ (y/n)").lower()
        if delete_files == "y":
            return True
        elif delete_files == "n":
            return False
        else:
            print("yかnで答えてください")

In [57]:
# 拡張画像ファイルの一斉削除
import os
import glob

delete_files_folders = [
    "chara0_aug",
    "chara1_aug",
    "chara2_aug",
    "chara3_aug",
    "test",
]
delete_nums = 0
if ask_delete_aug_files():
    for folder in delete_files_folders:
        file_list = glob.glob(f"{CHARACTER_PATH}/{folder}/*")
        delete_nums += len(file_list)
        for file in file_list:
            os.remove(file)
else:
    print("削除しませんでした")

print(f"削除したファイル数: {delete_nums}")

拡張画像を削除しますか？ (y/n)n
削除しませんでした
削除したファイル数: 0
